# SC4001 Phase 5b: does widening the ESM projection restore target ordering?

Accelerator **GPU (T4 x2 or P100)**. Internet is **not** needed: the frozen ESM-2 embeddings
ship in the attached dataset `isaacwjk21/sc4001-project` alongside the code and tables, so no
GPU time goes on recomputing them.

## The one question this kernel answers

The first ESM run (`sc4001-phase5-esm-coldtarget`) improved accuracy and widened the spread of
its predictions, but its **within-ligand concordance did not move**: 0.6867 against the CNN's
0.6789 on DAVIS, inside seed noise. Wider predictions that are not better ordered.

That result is confounded by a choice made for comparison cleanliness rather than for the
model. The frozen embedding is 1280-wide and was projected through `Linear(1280, 96)`, because
96 is the CNN branch's output width and keeping it identical left the fusion width and the whole
regression head unchanged between arms. It is also a **13x compression applied before the head
ever sees the embedding**, so a flat within-ligand CI may be the bottleneck rather than the
protein language model.

This kernel varies exactly that width and nothing else: 256, 512 and 1280 (no compression at
all), three seeds each, DAVIS cold-target only. Everything else matches the 96-wide arm --
same frozen embeddings, same drug CNN, same 1024/1024/512 head, same 100 epochs with patience
equal to max-epochs.

## What each outcome means

* **Within-ligand CI rises with width.** The bottleneck was ours. The Phase 3 reading survives
  and strengthens: the deficit is representational, and a protein language model does address
  it once it is not crushed to 96 numbers. The next run, DANN on this branch, becomes the test
  that matters, because alignment would then have a representation worth aligning.
* **Within-ligand CI stays flat at every width.** The bottleneck was not the projection.
  Mean-pooling over up to 1022 residues is then the prime suspect, since binding is local, and
  the honest write-up says a frozen whole-chain embedding does not separate unseen kinases
  however much width it is given.

Either way the claim in `update.md` section 3 stops being "we cannot tell".

## Guard rails

* Width is selected on **validation** MSE, pooled over the three seeds. Test metrics are
  reported for every width but never used to choose.
* Each width writes to its own output root (`outputs/esm-proj<width>/...`), so the 96-wide
  results already pulled back are untouched and remain reproducible.
* Wall-clock guard at 40 min. The 96-wide arm cost 1.4 min per DAVIS run, so 9 runs should cost
  roughly 15 min; the guard exists for the case where a wider head is much slower than that.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())
KERNEL_STARTED = time.time()


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# This kernel needs the two modules added for Phase 5 and the extra split columns.
required = ["esm_embed.py", "kmer_baseline.py"]
present = {p.name for p in (project / "src").iterdir()}
missing = [name for name in required if name not in present]
if missing:
    raise SystemExit("Attached dataset predates Phase 5; missing src modules: " + str(missing)
                     + ". Push a new dataset version before running this kernel.")
print("phase 5 modules OK")


In [ ]:
# The frozen ESM-2 embeddings come from the attached dataset: they were computed once by
# sc4001-phase5-esm-coldtarget and uploaded, so nothing is downloaded or recomputed here and
# this kernel needs no internet. src/esm_embed.py is unchanged and still reproduces them.

# Verify every embedding file against its processed target table before training reads it.
import numpy as np
import pandas as pd

processed = project / "data" / "processed"
EXPECTED = {"davis": "davis_targets", "kiba": "kiba_targets", "external": "external_kinases"}
for source, table_stem in EXPECTED.items():
    path = processed / (source + "_esm.npz")
    if not path.exists():
        raise SystemExit("Missing " + str(path))
    with np.load(path, allow_pickle=False) as arrays:
        embeddings = arrays["prot_embeddings"]
    stem = processed / (table_stem + ".csv")
    table = pd.read_csv(stem if stem.exists() else stem.with_suffix(".csv.gz"), usecols=[0])
    if embeddings.shape != (len(table), 1280):
        raise SystemExit(source + " embeddings are " + str(embeddings.shape)
                         + ", expected " + str((len(table), 1280)))
    if not np.isfinite(embeddings).all():
        raise SystemExit(source + " embeddings contain a nonfinite value.")
    norms = np.linalg.norm(embeddings, axis=1)
    duplicates = len(table) - len(np.unique(embeddings, axis=0))
    print("%-9s %s  L2 norm min %.3f mean %.3f max %.3f  duplicate rows %d"
          % (source, embeddings.shape, norms.min(), norms.mean(), norms.max(), duplicates),
          flush=True)
    if duplicates:
        print("   NOTE: identical embeddings mean two targets are indistinguishable to the "
              "protein branch. DAVIS has 423 distinct sequences, so this should be 0.",
              flush=True)


In [ ]:
# Three projection widths against the already-measured 96-wide arm. DAVIS cold-target only.
#
# 100 epochs with --patience equal to --max-epochs, matching the 96-wide arm exactly, so the
# stopping rule cannot account for any difference between widths.
ROOT = "/kaggle/working/outputs"
WIDTHS = (256, 512, 1280)
SEEDS = (4001, 4002, 4003)
GRID = [(width, seed) for width in WIDTHS for seed in SEEDS]
EPOCHS = 100

# The 96-wide arm measured 1.4 min per DAVIS run. A wider projection adds parameters only in the
# projection and the head's first layer, so start from 2.5 min: conservative, not optimistic.
PRIOR_SECONDS = 150.0
BUDGET_SECONDS = 40 * 60

status, skipped = [], []
for width, seed in GRID:
    elapsed = time.time() - KERNEL_STARTED
    done = [row[3] for row in status if row[0] == width and row[2] == 0]
    projected = max(done) if done else PRIOR_SECONDS
    if elapsed + projected > BUDGET_SECONDS:
        skipped.append((width, seed, round(projected / 60, 1)))
        print("SKIPPING proj" + str(width) + " seed" + str(seed) + ": projected "
              + format(projected / 60, ".1f") + " min would exceed the "
              + format(BUDGET_SECONDS / 60, ".0f") + " min guard ("
              + format(elapsed / 60, ".1f") + " min used)", flush=True)
        continue
    print()
    print("===== esm proj" + str(width) + " davis split_cold_target seed" + str(seed)
          + " (" + str(EPOCHS) + " epochs, no early stop) =====", flush=True)
    t0 = time.time()
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", "davis", "--split", "split_cold_target",
         "--seed", str(seed), "--protein-encoder", "esm", "--esm-proj-dim", str(width),
         "--max-epochs", str(EPOCHS), "--patience", str(EPOCHS), "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT), check=False,
    )
    wall = time.time() - t0
    status.append((width, seed, run.returncode, wall))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f") + " min ("
          + format(wall / EPOCHS, ".2f") + " s/epoch) -----", flush=True)

print()
print("Grid wall time", format((time.time() - KERNEL_STARTED) / 60, ".1f"), "min since kernel start")
for width, seed, rc, wall in status:
    print("  proj%-5d seed%d rc=%d %6.1f min  %5.2f s/epoch"
          % (width, seed, rc, wall / 60, wall / EPOCHS))
failed = [row for row in status if row[2] != 0]
print("FAILED RUNS:", failed if failed else "none")
print("SKIPPED FOR BUDGET:", skipped if skipped else "none")
if failed:
    print("A failed width cannot be read as evidence either way. Check its traceback above.")


In [ ]:
# Validation selects the width; test is reported for every width but never used to choose.
import statistics as st

# The 96-wide arm, already run and pulled back (outputs/esm/davis/split_cold_target/seed*).
BASE96 = {"val": [0.3743, 0.4337, 0.4186],
          "test_all_mse": [0.4194, 0.4011, 0.3857], "test_all_ci": [0.8064, 0.8076, 0.8125],
          "test_unc_mse": [1.1125, 0.9842, 0.9341], "test_unc_ci": [0.6696, 0.6602, 0.6690]}
# Equal-compute CNN arm and the non-neural k-mer baseline, for context only.
CNN = {"test_all_mse": [0.3991, 0.4208, 0.3775], "test_all_ci": [0.7901, 0.7879, 0.7874],
       "test_unc_mse": [1.2087, 1.1723, 0.9901], "test_unc_ci": [0.6454, 0.6387, 0.6665]}
KMER = {"test_all_mse": 0.3516, "test_all_ci": 0.8385,
        "test_unc_mse": 0.6875, "test_unc_ci": 0.7138}

runs = {}
for path in sorted(Path(ROOT).glob("esm-proj*/davis/**/metrics.json")):
    payload = json.loads(path.read_text())
    width = int(json.loads((path.parent / "config.json").read_text())["esm_proj_dim"])
    runs.setdefault(width, []).append(payload)

rows = {96: BASE96}
for width in sorted(runs):
    payloads = sorted(runs[width], key=lambda row: row["seed"])
    rows[width] = {
        "val": [p["val"]["all"]["mse"] for p in payloads],
        "test_all_mse": [p["test"]["all"]["mse"] for p in payloads],
        "test_all_ci": [p["test"]["all"]["ci"] for p in payloads],
        "test_unc_mse": [p["test"]["uncensored"]["mse"] for p in payloads],
        "test_unc_ci": [p["test"]["uncensored"]["ci"] for p in payloads],
        "best_epoch": [p["best_epoch"] for p in payloads],
        "seeds": [p["seed"] for p in payloads],
    }
    print("proj%-5d seeds %s best epochs %s" % (width, rows[width]["seeds"],
                                                rows[width]["best_epoch"]))

print()
print("%-7s %-5s %-16s %-16s %-16s %-16s" % ("width", "n", "val MSE", "test MSE",
                                             "test CI", "uncens MSE"))
for width in sorted(rows):
    row = rows[width]
    print("%-7d %-5d %-16.4f %-16.4f %-16.4f %-16.4f"
          % (width, len(row["val"]), st.mean(row["val"]), st.mean(row["test_all_mse"]),
             st.mean(row["test_all_ci"]), st.mean(row["test_unc_mse"])))
print("%-7s %-5d %-16s %-16.4f %-16.4f %-16.4f"
      % ("cnn96", 3, "-", st.mean(CNN["test_all_mse"]), st.mean(CNN["test_all_ci"]),
         st.mean(CNN["test_unc_mse"])))
print("%-7s %-5d %-16s %-16.4f %-16.4f %-16.4f"
      % ("kmer", 1, "-", KMER["test_all_mse"], KMER["test_all_ci"], KMER["test_unc_mse"]))

complete = [w for w in rows if len(rows[w]["val"]) == 3]
BEST = min(complete, key=lambda w: st.mean(rows[w]["val"]))
print()
print("Width selected on validation MSE:", BEST,
      "(val %.4f against 96-wide %.4f)" % (st.mean(rows[BEST]["val"]), st.mean(BASE96["val"])))
if BEST == 96:
    print("No width beats the 96-wide arm on validation. The projection was not the bottleneck,")
    print("and the next suspect is mean-pooling over the whole chain, not the width.")

# Paired by seed against the 96-wide arm, for every width that completed all three seeds.
for width in sorted(w for w in complete if w != 96):
    print()
    print("===== proj%d vs proj96, paired by seed, 2 df =====" % width)
    for label, key in (("test all MSE", "test_all_mse"), ("test all CI", "test_all_ci"),
                       ("uncens MSE", "test_unc_mse"), ("uncens CI", "test_unc_ci")):
        diffs = [a - b for a, b in zip(rows[width][key], BASE96[key])]
        mean = st.mean(diffs)
        sd = st.stdev(diffs)
        t = mean / (sd / len(diffs) ** 0.5) if sd > 0 else float("inf")
        signs = " ".join("+" if d > 0 else "-" for d in diffs)
        print("  %-13s %+.4f  t(2)=%+6.2f  seed signs %s" % (label, mean, t, signs))
    print("  Seed signs are proj%d minus proj96 for seeds 4001, 4002, 4003." % width)


In [ ]:
# The measurement the whole kernel exists for: within-ligand concordance, which did NOT move
# when the projection was 96 wide.
#
#   within-ligand CI   concordance inside each test drug's own measured test pairs.
#   predicted spread   predict each drug against EVERY cold test target, measured or not, take
#                      the sd, average over drugs. Exactly 0 for a model that cannot separate
#                      unseen targets.
# Both restricted to the matched ligand set: drugs with at least two measured test pairs and a
# non-zero measured spread (66 of 68 DAVIS drugs, measured spread 0.6391).
sys.path.insert(0, str(project))
import numpy as np
import pandas as pd
from src.config import Config
from src.data import load_esm_features, processed_csv, resolve_split_column
from src.model import DeepDTA

DATA = project / "data" / "processed"
REFERENCE = {"cnn96": {"ci": 0.6789, "spread": 0.3679, "ratio": 0.576},
             "esm96": {"ci": 0.6867, "spread": 0.3975, "ratio": 0.622},
             "kmer": {"ci": 0.7494, "spread": None, "ratio": 0.771}}
MEASURED = 0.6391
device = torch.device("cuda")


def pairwise_ci(y, prediction):
    """Return one drug's concordance index, prediction ties counted as half."""
    y, prediction = np.asarray(y, dtype=float), np.asarray(prediction, dtype=float)
    better = y[:, None] > y[None, :]
    if not better.any():
        return None
    gap = prediction[:, None] - prediction[None, :]
    return float(((gap[better] > 0).sum() + 0.5 * (gap[better] == 0).sum()) / better.sum())


def batched(model, drug_tokens, protein_features, drug_rows, protein_rows, batch=1024):
    """Predict for aligned drug/target row-index arrays."""
    out = []
    with torch.no_grad():
        for start in range(0, len(drug_rows), batch):
            d = drug_tokens[drug_rows[start:start + batch]].to(device)
            p = protein_features[protein_rows[start:start + batch]].to(device)
            out.append(model(d, p).detach().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

summary = {}
for path in sorted(Path(ROOT).glob("esm-proj*/davis/**/metrics.json")):
    run_dir = path.parent
    saved = json.loads((run_dir / "config.json").read_text())
    config = Config(**{k: v for k, v in saved.items() if k in Config.__dataclass_fields__})
    width = config.esm_proj_dim
    model = DeepDTA(saved["vocab_sizes"]["smiles"], saved["vocab_sizes"]["protein"], config).to(device)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device, weights_only=True))
    model.eval()

    pairs = pd.read_csv(processed_csv(DATA, "davis_pairs"),
                        dtype={"ligand_id": str, "target_id": str})
    ligands = pd.read_csv(processed_csv(DATA, "davis_ligands"), usecols=["ligand_id"],
                          dtype={"ligand_id": str})
    targets = pd.read_csv(processed_csv(DATA, "davis_targets"), usecols=["target_id"],
                          dtype={"target_id": str})
    ligand_row = dict(zip(ligands.ligand_id, range(len(ligands))))
    target_row = dict(zip(targets.target_id, range(len(targets))))
    test = pairs.loc[resolve_split_column(pairs, DATA, "davis", config.split) == "test"].copy()

    with np.load(DATA / "davis_tokens.npz", allow_pickle=False) as arrays:
        drug_tokens = torch.tensor(arrays["drug_tokens"], dtype=torch.long)
    protein_features = load_esm_features(DATA, "davis", len(targets), config.esm_dim)

    test["pred"] = batched(model, drug_tokens, protein_features,
                           test.ligand_id.map(ligand_row).to_numpy(),
                           test.target_id.map(target_row).to_numpy())
    measured = test.groupby("ligand_id").score.std(ddof=1)
    matched = sorted(measured[(measured > 0) & measured.notna()].index)
    cis = [v for drug in matched
           if (v := pairwise_ci(test.loc[test.ligand_id == drug, "score"],
                                test.loc[test.ligand_id == drug, "pred"])) is not None]

    test_targets = sorted(test.target_id.unique())
    grid_drugs = np.repeat([ligand_row[d] for d in matched], len(test_targets))
    grid_prots = np.tile([target_row[t] for t in test_targets], len(matched))
    grid = batched(model, drug_tokens, protein_features, grid_drugs, grid_prots)
    spreads = grid.reshape(len(matched), len(test_targets)).std(axis=1, ddof=1)

    summary.setdefault(width, []).append((config.seed, float(np.mean(cis)),
                                          float(spreads.mean())))
    print("proj%-5d seed%d  within-ligand CI %.4f  predicted spread %.4f  ratio %.3f  "
          "drugs at spread 0: %d of %d"
          % (width, config.seed, np.mean(cis), spreads.mean(), spreads.mean() / MEASURED,
             (spreads < 1e-9).sum(), len(matched)), flush=True)

print()
print("%-8s %-5s %-18s %-18s %-10s" % ("width", "n", "within-ligand CI", "pred spread", "ratio"))
for width in sorted(summary):
    rows_w = summary[width]
    ci = sum(r[1] for r in rows_w) / len(rows_w)
    spread = sum(r[2] for r in rows_w) / len(rows_w)
    print("%-8d %-5d %-18.4f %-18.4f %-10.3f" % (width, len(rows_w), ci, spread, spread / MEASURED))
for name, ref in REFERENCE.items():
    spread = "-" if ref["spread"] is None else "%.4f" % ref["spread"]
    print("%-8s %-5s %-18.4f %-18s %-10.3f" % (name, "ref", ref["ci"], spread, ref["ratio"]))

print()
for width in sorted(summary):
    rows_w = summary[width]
    ci = sum(r[1] for r in rows_w) / len(rows_w)
    print("proj%-5d within-ligand CI %+.4f against esm96, %+.4f against cnn96"
          % (width, ci - REFERENCE["esm96"]["ci"], ci - REFERENCE["cnn96"]["ci"]))
print()
print("Read it this way. A rise of more than about 0.02 over esm96 (0.6867), with all three")
print("seeds in the same direction, means the 96-wide projection was the bottleneck and a")
print("protein language model does separate unseen kinases once it is not crushed. A flat")
print("result at every width means the projection was never the constraint, and whole-chain")
print("mean-pooling is the next thing to vary. Per-seed CI on the 96-wide arm was")
print("0.6778 / 0.6692 / 0.7132, so that is the noise this has to clear.")
